In [12]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

raw_text = Path("output/doc_0.md").read_text(encoding="utf-8")

In [13]:
from openai import OpenAI
import re
import json
import os

# ── 配置区：全部从环境变量读取，本地开发在 .env 里设置即可 ──────────────────────
#
#   必填：
#     LLM_API_KEY     你的 API Key
#
#   可选（有默认值）：
#     LLM_BASE_URL    不填则走 OpenAI 官方地址；填 DeepSeek 就写 https://api.deepseek.com
#     LLM_MODEL       默认 gpt-4o，DeepSeek 就填 deepseek-chat

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ.get("LLM_BASE_URL") or None,
)
MODEL = os.environ.get("LLM_MODEL", "gpt-4o")

In [14]:
# ── Step 0：提取标题树，预览文档结构 ─────────────────────────────────────────

def extract_headings(text: str) -> list[dict]:
    """
    扫描全文，提取每个标题的层级、文本、行号。
    返回格式：[{"level": 2, "text": "实习经历", "line": 12}, ...]
    """
    headings = []
    for line_no, line in enumerate(text.splitlines()):
        match = re.match(r"^(#{1,6})\s+(.+)", line)
        if match:
            headings.append({
                "level": len(match.group(1)),
                "text":  match.group(2).strip(),
                "line":  line_no,
            })
    return headings


headings = extract_headings(raw_text)

print(f"一共找到 {len(headings)} 个标题：\n")
for h in headings:
    print("  " * (h["level"] - 1) + f"- H{h['level']}: {h['text']}")

一共找到 5 个标题：

- H1: 范逸
  - H2: 教育背景
  - H2: 实习经历
  - H2: 美的集团·中央研究院
  - H2: 项目经历


In [15]:
# ── Step 1：LLM 审查标题树，修正层级错误 ──────────────────────────────────────
#
# 不做规则预筛，直接交给 LLM 凭语义判断，对任意文档类型都成立。

def llm_verify_headings(all_headings: list[dict]) -> list[dict]:
    """
    把完整标题树喂给 LLM，返回需要修正的标题列表。
    格式：[{"text": "...", "correct_level": 3}, ...]
    """
    if not all_headings:
        return []

    tree_str = "\n".join(
        f"{'  ' * (h['level'] - 1)}H{h['level']}: {h['text']}"
        for h in all_headings
    )

    prompt = f"""下面是一份从 PDF 解析出来的 Markdown 标题树。
PDF 解析器有时会把标题层级识别错——常见错误是某个标题在语义上应该是上一个标题的子项，却被识别成了同级。

标题树：
{tree_str}

请仔细阅读这棵树，找出所有语义层级与当前 H 级别不符的标题。
判断标准：只看语义——如果一个标题描述的是某个上级标题下的具体实例或细节，它就应该是子级。

只返回需要修正的标题，格式为 JSON 数组，不要输出任何其他内容：
[
  {{"text": "标题原文", "correct_level": 正确的层级数字}},
  ...
]

如果整棵树层级都正确，返回空数组 []。"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    raw = resp.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", raw, re.DOTALL)
        return json.loads(match.group()) if match else []


corrections = llm_verify_headings(headings)
correction_map = {c["text"]: c["correct_level"] for c in corrections}

# 打印纠正明细
if corrections:
    print(f"\n{'='*60}")
    print(f"LLM 共发现 {len(corrections)} 处标题层级错误：")
    for c in corrections:
        original = next((h for h in headings if h["text"] == c["text"]), None)
        original_level = original["level"] if original else "?"
        print(f"  H{original_level} → H{c['correct_level']}  |  {c['text']}")
    print(f"{'='*60}\n")
else:
    print("LLM 未发现标题层级错误，无需修正。")


LLM 共发现 1 处标题层级错误：
  H2 → H3  |  美的集团·中央研究院



In [16]:
# ── Step 1b：把修正应用到原文 ─────────────────────────────────────────────────

def apply_corrections(text: str, correction_map: dict) -> str:
    """
    按行扫描原文，遇到需要修正的标题就替换成正确层级的 '#'，其余行原样保留。
    """
    fixed_lines = []
    for line in text.splitlines():
        match = re.match(r"^(#{1,6})\s+(.+)", line)
        if match:
            heading_text = match.group(2).strip()
            if heading_text in correction_map:
                new_level = correction_map[heading_text]
                line = "#" * new_level + " " + heading_text
        fixed_lines.append(line)
    return "\n".join(fixed_lines)


fixed_text = apply_corrections(raw_text, correction_map)

In [17]:
# ── Step 2：切割 chunk，同时识别哪些 chunk 含表格 ──────────────────────────────
#
# 切割逻辑本身不变，只是在生成 chunk 时顺手标记 has_table。
# 含表格的 chunk 后续走 JSON 提取，不进 embedding。

def split_into_chunks(text: str) -> list[dict]:
    """
    按标题切割文本，每个标题就是一个切割点。
    每个 chunk 额外带一个 has_table 字段，标记内容里是否含有 HTML 表格。
    """
    lines = text.splitlines()
    chunks = []
    current_chunk_lines = []
    current_heading = None
    heading_stack: list[dict] = []

    def flush(stack):
        """把当前积累的行打包成一个 chunk，顺手检测是否含表格。"""
        content = "\n".join(current_chunk_lines).strip()
        if not content:
            return None
        return {
            "heading_path": " > ".join(h["text"] for h in stack),
            "content":      content,
            # <table 出现就认定含表格，简单可靠
            "has_table":    "<table" in content,
        }

    for line in lines:
        match = re.match(r"^(#{1,6})\s+(.+)", line)

        if not match:
            current_chunk_lines.append(line)
            continue

        level       = len(match.group(1))
        heading_text = match.group(2).strip()

        if current_heading is not None:
            chunk = flush(heading_stack)
            if chunk:
                chunks.append(chunk)
            current_chunk_lines = []

        while heading_stack and heading_stack[-1]["level"] >= level:
            heading_stack.pop()
        heading_stack.append({"level": level, "text": heading_text})

        current_heading = {"level": level, "text": heading_text}
        current_chunk_lines.append(line)

    # 最后一个 chunk
    chunk = flush(heading_stack)
    if chunk:
        chunks.append(chunk)

    return chunks


chunks = split_into_chunks(fixed_text)

text_chunks  = [c for c in chunks if not c["has_table"]]
table_chunks = [c for c in chunks if c["has_table"]]

print(f"共 {len(chunks)} 个 chunk：{len(text_chunks)} 个纯文本，{len(table_chunks)} 个含表格")

共 5 个 chunk：5 个纯文本，0 个含表格


In [18]:
# ── Step 3：对含表格的 chunk，用 LLM 提取结构化 JSON ──────────────────────────
#
# 表格不进 embedding，单独存成 JSON，查询时走精确匹配。
# LLM 负责把 HTML 表格转成字段清晰的 JSON 数组，
# 字段名由 LLM 根据表格内容自行推断，不做硬编码。

def extract_table_as_json(chunk: dict) -> list[dict]:
    """
    把含表格的 chunk 喂给 LLM，提取成 JSON 数组。
    每一行表格对应一个 JSON 对象，字段名从表头推断。
    返回的每条记录额外带上来源信息，方便溯源。
    """
    prompt = f"""下面是一段包含 HTML 表格的文本，来自文档章节：{chunk['heading_path']}

{chunk['content']}

请把表格内容提取成 JSON 数组，要求：
1. 每行表格数据对应一个 JSON 对象
2. 字段名用英文，根据表头语义自行命名，简洁清晰
3. 字段值保留原文，不要改写
4. 只返回 JSON 数组，不要输出任何其他内容

"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    raw = resp.choices[0].message.content.strip()
    try:
        rows = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        rows = json.loads(m.group()) if m else []

    # 每条记录带上来源，查询结果可以溯源到具体章节
    for row in rows:
        row["_source_heading"] = chunk["heading_path"]

    return rows


# 对所有含表格的 chunk 逐一提取
all_table_records = []
for chunk in table_chunks:
    records = extract_table_as_json(chunk)
    all_table_records.extend(records)
    print(f"\n章节: {chunk['heading_path']}")
    print(f"提取到 {len(records)} 条记录，示例：")
    print(json.dumps(records[0], ensure_ascii=False, indent=2) if records else "  (空)")

In [19]:
# ── Step 4：预览结果 ──────────────────────────────────────────────────────────

print("\n── 纯文本 chunks（将进入 embedding）──")
for i, chunk in enumerate(text_chunks):
    print(f"\n{'='*60}")
    print(f"Chunk {i+1}  |  路径: {chunk['heading_path']}")
    print(f"{'='*60}")
    print(chunk["content"][:300], "..." if len(chunk["content"]) > 300 else "")

print("\n\n── 表格 JSON（将走精确匹配）──")
print(json.dumps(all_table_records, ensure_ascii=False, indent=2))


── 纯文本 chunks（将进入 embedding）──

Chunk 1  |  路径: 范逸
# 范逸

15601792659 | 15601792659@163.com | 男 | 中共党员

<div style="text-align: center;"><img src="imgs/img_in_image_box_973_21_1093_163.jpg" alt="Image" width="10%" /></div> 

Chunk 2  |  路径: 范逸 > 教育背景
## 教育背景

新南威尔士大学·信息技术（数据科学与工程）

碩士

主修课程：深度学习与神经网络、机器学习、统计推断、图算法与图神经网络、大数据技术

河南理工大学·计算机科学与技术

2023.09 – 2025.10

学士

2019.09 – 2023.07

两次获得国家励志奖学金、一次孙越崎一等奖学金、两次三好学生荣誉称号，英语 CET-6 542 分，IELTS 6.5 分 

Chunk 3  |  路径: 范逸 > 实习经历
## 实习经历

北京抖音信息服务有限公司

推荐算法

2025.05 - 2025.07

➢ 精排任务需同时优化完播率与观看时长，两目标在训练中存在冲突，且时长偏差使长视频更难完播，分发结果对短视频偏置明显

在兼顾完播率与观看时长的前提下缓解多任务冲突，降低强统计特征过拟合，提升长视频与弱历史样本的泛化能力。

从共享底层 DNN 加双塔头基线出发，引入 MMoE 进行任务表示分离；通过特征重要性与分桶实验定位全局热度等稠密特征过强导致 ID 与内容特征贡献受抑；在 MMoE 上叠加 SENet 重加权与双线性交叉，增强热度内容上下文 ID 特征的有效交互。

离线结果达到完播率 AU ...

Chunk 4  |  路径: 范逸 > 实习经历 > 美的集团·中央研究院
### 美的集团·中央研究院

研发工程师

2024.12 - 2025.02

空调实验测试端在初期存在实验参数主要依赖手工录入，实时监控与历史查询链路分散，页面长时间运行后易出现图表错位和性能下降三类问题；同时工况智能分析能力尚未落地。

在既有系统上完成“参数配置-实验执行-实时监控-历史导出”落地，并推进工况智能分析对话模块的前端原型验证。



In [23]:
# ── Step 5：embedding + 存入 Qdrant ──────────────────────────────────────────
#
# embedding.py 里定义了可插拔的 embedder 接口和 Qdrant 入库逻辑。
# 这里只需要三行：选 embedder、入库、完成。
#
# 切换模型只需改 get_embedder() 的参数，或者设置环境变量 EMBEDDER=openai/bge，
# 其余代码不用动。
#
# 前置条件：本地 Qdrant 已启动
#   docker run -p 6333:6333 qdrant/qdrant



from embedding import get_embedder, upsert_chunks

# doc_id 用文件名，多文档入库时方便按文档过滤
doc_id = Path("output/doc_0.md").stem

# 语言选项："en" / "zh" / "mixed" / "auto"
# 也可以在 .env 里设置 EMBEDDER=openai 或 EMBEDDER=bge 来强制覆盖
embedder = get_embedder(doc_language="en")

upsert_chunks(
    chunks=text_chunks,
    embedder=embedder,
    collection_name="documents",
    doc_id=doc_id,
)

collection 已存在，跳过创建: documents
正在 embedding 5 个 chunks...
已写入 5 条向量到 collection: documents


In [37]:
from qdrant_client import QdrantClient

q = QdrantClient(host="localhost", port=6333, proxy=None, trust_env=False)

pts = q.retrieve(
    collection_name="documents",
    ids=["d199a65a-c367-4c41-92fe-79d88d15fc70"],  # 换成真实的 point id
    with_vectors=True,
    with_payload=True,
)

vec = pts[0].vector
if isinstance(vec, dict):
    vec = vec.get("default") or next(iter(vec.values()))

print("dim:", len(vec))
print("first10:", vec[:10])   # 前10维
print("full:", vec)           # 全量向量


dim: 1536
first10: [0.008734078, 0.031427424, 0.044425566, 0.0008395585, 0.007856856, -0.0005468334, -0.0039856383, 0.047842916, -0.008085697, -0.006788934]
full: [0.008734078, 0.031427424, 0.044425566, 0.0008395585, 0.007856856, -0.0005468334, -0.0039856383, 0.047842916, -0.008085697, -0.006788934, 0.019909121, -0.011739517, -0.057576265, -0.028666085, 0.018124167, 0.0039741965, -0.053304575, -0.030100152, -0.02964247, -0.003285768, 0.025675902, 0.0058697583, 0.041313335, 0.0063846493, -0.0025611063, -0.04329662, -0.0031160444, 0.056843974, -0.016171394, -0.017285084, 0.017330851, -0.01788007, -0.0038807532, 0.030786673, 0.025019892, 0.008749334, 0.04216767, 0.0021777987, -0.029962847, 0.022884049, 0.0075441077, -0.009840141, -0.013021023, 0.049185447, 0.027018433, 0.020565132, -0.0541284, 0.04894135, 0.010732618, 0.0566609, -0.027506625, 0.012959999, -0.00067364913, 0.015599293, -0.041709993, 0.0135321, 0.003762519, 0.009885909, -0.027964307, -0.0027842259, 0.027674442, -0.030206943,

In [35]:
import importlib, retrieval
importlib.reload(retrieval)


<module 'retrieval' from '/Users/felix/Desktop/powerful-rag/notebooks/retrieval.py'>

In [36]:
# ── Step 6：Hybrid 检索测试 ───────────────────────────────────────────────────
#
# 三路并行：dense 向量 + 表格精确匹配，RRF 融合排序。
# sparse BM42 需要在 Qdrant 里单独建稀疏向量索引，后续补充。

from retrieval import hybrid_search, print_results

query = "范逸有哪些项目经历"

results = hybrid_search(
    query=query,
    embedder=embedder,              # Step 5 里初始化的 embedder 实例
    collection_name="documents",
    table_records=all_table_records, # Step 3 里提取的表格数据
    top_k=5,
)

print_results(results)


#1  来源: vector  |  Final: 0.1221  |  RRF: 0.0296  |  路径: 范逸 > 项目经历
分数构成: final = rrf(0.0296) + lexical_bonus(0.0925) [heading_overlap=0.5, content_overlap=0.375]
RRF构成: vector@rank7:0.014925 + vector@rank8:0.014706
## 项目经历

Orchestrix AI（n8n-like 可视化 AI 工作流平台）

2025.09–2025.11

• 项目背景：工作流平台的核心价值是将跨系统、多步骤、异步流程标准化，降低人工操作成本，提高流程的稳定性、可追溯性和复用效率。计划通过完整自研验证其底层机制，提高自己对 AI 应用工程化落地的理解深度。

- 核心任务：以独立项目方式实现一个 AI 赋能的、类似于 n8n 的工作流平台，覆盖编排、执行、观测、调度和外部能力集成，形成可运行的端到端闭环，沉淀可复用的工程方法。

· 项目内容:

1. 从 0 到 1 完成了平台核心架构与实现，基于 API Route  ...

#2  来源: vector  |  Final: 0.073  |  RRF: 0.0305  |  路径: 范逸 > 实习经历
分数构成: final = rrf(0.0305) + lexical_bonus(0.0425) [heading_overlap=0.25, content_overlap=0.125]
RRF构成: vector@rank5:0.015385 + vector@rank6:0.015152
## 实习经历

北京抖音信息服务有限公司

推荐算法

2025.05 - 2025.07

➢ 精排任务需同时优化完播率与观看时长，两目标在训练中存在冲突，且时长偏差使长视频更难完播，分发结果对短视频偏置明显

在兼顾完播率与观看时长的前提下缓解多任务冲突，降低强统计特征过拟合，提升长视频与弱历史样本的泛化能力。

从共享底层 DNN 加双塔头基线出发，引入 MMoE 进行任务表示分离；通过特征重要性与分桶实验定位全局热度等稠密特征过强导致 ID 与内容特征贡献受抑；在 MMoE 上叠加 SENet 重加权与双线性交